In [32]:
import torch
from datasets import load_dataset
from transformers import T5Tokenizer, T5ForConditionalGeneration
from translation_dataset import create_dataloader, TranslationDataset
from prefix_t5_model import PrefixT5
from trainer import Trainer, TrainingHyperParams, load_model
from prediction import generate_translation
from configs import CHECKPOINT, DATASET, BATCH_SIZE

In [33]:
t5_tokenizer = T5Tokenizer.from_pretrained(CHECKPOINT)
t5_model = T5ForConditionalGeneration.from_pretrained(CHECKPOINT)

## Data Preprocessing

In [34]:
# Load dataset
dataset = load_dataset(DATASET, "en-fr")['train']

# Shuffle the dataset
dataset = dataset.shuffle(seed=42)

# Split the dataset into train, valid, test (train_size=80%, valid_size=10%, test_size=10%)
train_valid_dataset, test_dataset = torch.utils.data.random_split(dataset, [0.9, 0.1])
train_dataset, valid_dataset = torch.utils.data.random_split(train_valid_dataset, [0.9, 0.1])

In [35]:
train_translation_dataset = TranslationDataset(
    data=train_dataset,
    tokenizer=t5_tokenizer,
    shift_right_func=t5_model._shift_right
)

valid_translation_dataset = TranslationDataset(
    data=valid_dataset,
    tokenizer=t5_tokenizer,
    shift_right_func=t5_model._shift_right
)

test_translation_dataset = TranslationDataset(
    data=test_dataset,
    tokenizer=t5_tokenizer,
    shift_right_func=t5_model._shift_right
)


In [36]:
train_loader = create_dataloader(
    dataset=train_translation_dataset, 
    batch_size=BATCH_SIZE
)

valid_loader = create_dataloader(
    dataset=valid_translation_dataset, 
    batch_size=BATCH_SIZE
)

test_loader = create_dataloader(
    dataset=test_translation_dataset, 
    batch_size=BATCH_SIZE, 
)

In [37]:
print(f"Number of batches in train: {len(train_loader)}")
print(f"Number of batches in valid: {len(valid_loader)}")
print(f"Number of batches in test: {len(test_loader)}")

Number of batches in train: 6434
Number of batches in valid: 715
Number of batches in test: 795


In [38]:
# Get a sample batch
sample_batch = next(iter(train_loader))
    
print("\nSample batch shapes:")
for key, value in sample_batch.items():
    print(f"{key}: {value.shape}")

# Decode a sample from the batch
print("\n")
print("Input:", t5_tokenizer.decode(sample_batch['input_ids'][0].tolist(), skip_special_tokens=True))
print("Target:", t5_tokenizer.decode(sample_batch['labels'][0].tolist(), skip_special_tokens=True))


Sample batch shapes:
input_ids: torch.Size([16, 128])
attention_mask: torch.Size([16, 128])
labels: torch.Size([16, 128])
decoder_input_ids: torch.Size([16, 128])


Input: translate English to French: 'I have to speak to you: I must speak to you, tonight; when one o'clock strikes, be in the garden.
Target: « J’ai besoin de vous parler : il faut que je vous parle, ce soir ; au moment où une heure après minuit sonnera, trouvez-vous dans le jardin.


## Prefix T5 Model

In [39]:
prefixed_t5_model = PrefixT5(
    t5_model=t5_model, prefix_length=10, prefix_projection_size=512
)

## Training

In [40]:
sample_data = dataset.select(range(50))
sample_valid = dataset.select(range(50, 70))
sample_trans_data = TranslationDataset(
    data=sample_data,
    tokenizer=t5_tokenizer,
    shift_right_func=t5_model._shift_right
)
sample_loader = create_dataloader(sample_trans_data)

sample_valid_trans_data = TranslationDataset(
    data=sample_valid,
    tokenizer=t5_tokenizer,
    shift_right_func=t5_model._shift_right
)
sample_valid_loader = create_dataloader(sample_valid_trans_data)

In [42]:
training_params = TrainingHyperParams(
    num_epochs=1, 
    batch_size=16, 
    learning_rate=0.001, 
    warmup_steps=100, 
    max_grad_norm=1,
)

trainer = Trainer(
    model=prefixed_t5_model,
    tokenizer=t5_tokenizer,
    train_loader=train_loader,
    valid_loader=valid_loader,
    test_loader=test_loader,
    args=training_params
)

In [ ]:
train_loss, val_loss = trainer.train()

In [12]:
saved_model = load_model(base_model=t5_model, save_path="models/prefix_t5-num_epochs_1-batch_size_16-learning_rate_0.001-warmup_steps_100-max_grad_norm_1-device_cpu.pth")
saved_model 

Model successfully loaded from models/prefix_t5-num_epochs_1-batch_size_16-learning_rate_0.001-warmup_steps_100-max_grad_norm_1-device_cpu.pth


PrefixT5(
  (t5): T5ForConditionalGeneration(
    (shared): Embedding(32128, 512)
    (encoder): T5Stack(
      (embed_tokens): Embedding(32128, 512)
      (block): ModuleList(
        (0): T5Block(
          (layer): ModuleList(
            (0): T5LayerSelfAttention(
              (SelfAttention): T5Attention(
                (q): Linear(in_features=512, out_features=512, bias=False)
                (k): Linear(in_features=512, out_features=512, bias=False)
                (v): Linear(in_features=512, out_features=512, bias=False)
                (o): Linear(in_features=512, out_features=512, bias=False)
                (relative_attention_bias): Embedding(32, 8)
              )
              (layer_norm): T5LayerNorm()
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (1): T5LayerFF(
              (DenseReluDense): T5DenseActDense(
                (wi): Linear(in_features=512, out_features=2048, bias=False)
                (wo): Linear(in_features=2048,

In [30]:
import torch

from transformers import T5Tokenizer


@torch.no_grad()
def generate_translation(
    model: torch.nn.Module,
    tokenizer: T5Tokenizer,
    input_text, max_length: int = 1, 
    device: str = "cuda" if torch.cuda.is_available() else "cpu",
) -> str:
        
    """Generate and print sample translations"""
    model.eval().to(device)

    inputs = tokenizer(input_text, return_tensors="pt", truncation=True, padding=True).to(device)
    input_ids = inputs["input_ids"]
    attention_mask = inputs["attention_mask"]

    decoder_input_ids = torch.full(
        (input_ids.size(0), 1),  # batch_size x 1
        tokenizer.pad_token_id,  # Start with <pad> (can be adjusted for specific tasks)
        device=device,
    )

    generated_ids = []

    for _ in range(max_length):
        
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=None,
            decoder_input_ids=decoder_input_ids,
        )

        # print(outputs.logits[:, -1, :].shape)

        logits = outputs.logits[:, -1, :]
        next_token_id = logits.argmax(dim=-1)
        print(tokenizer.decode(next_token_id))
    #     generated_ids.append(next_token_id)

    #     decoder_input_ids = torch.cat([decoder_input_ids, next_token_id.unsqueeze(-1)], dim=-1)

    #     if next_token_id.item() == tokenizer.eos_token_id:
    #         break

    # generated_text = tokenizer.decode(torch.cat(generated_ids), skip_special_tokens=True)
    # return generated_text


In [31]:
res = generate_translation(
    model=saved_model,
    tokenizer=t5_tokenizer, 
    input_text="Hello, my name is Zeinab",
    max_length=2,
)
res

</s>
</s>


In [15]:
print("Pad Token ID:", t5_tokenizer.pad_token_id)
print("EOS Token ID:", t5_tokenizer.eos_token_id)
print("BOS Token ID:", t5_tokenizer.bos_token_id)  # If applicable


Pad Token ID: 0
EOS Token ID: 1
BOS Token ID: None
